## Notebook 03_gold_modelagem — Gold: Modelagem Dimensional

#### Objetivo
Consumir a camada Silver e materializar a camada Gold: 

- um **star schema** (modelagem dimensional em estrela) com o ciclo de vida completo dos processos; 
- um **mart analítico** que alimenta as 9 perguntas do MVP; e
- a governança do catálogo (dicionário de dados + QA com regression check).

#### O que é o star schema (e por que aqui)
Modelo dimensional clássico de Kimball: uma **tabela fato** central (fatos = eventos mensuráveis, números, datas) cercada por **tabelas dimensão** (contextos descritivos pelos quais se filtra e agrupa). O nome vem do desenho: no diagrama, as dimensões irradiam da fato como pontas de estrela.

**Por que separar fato de dimensão:**
- A fato fica **enxuta e numérica** — agregações (SUM, COUNT, AVG) são rápidas e o grão fica explícito (aqui: 1 linha por processo).
- As dimensões guardam o **contexto descritivo** (classe, órgão, tempo) — mudanças de rótulo não poluem a fato, e o mesmo contexto serve a várias métricas sem duplicação.
- É o padrão que ferramentas de BI esperam.

## O desenho implementado
| Tabela | Papel | Grão |
|---|---|---|
| `dim_tempo` | Calendário do corpus (1995–2027) | 1 linha por dia |
| `dim_classe_processual` | Classes da TPU do CNJ | 1 linha por classe |
| `dim_orgao_julgador` | Órgãos julgadores, dedup por (código, tribunal) | 1 linha por órgão×tribunal |
| `dim_processo` | 225.556 processos, todos da silver após 1/01/1995  | 1 linha por processo |
| `fato_tramitacao` | **Fato central**: ciclo de vida + durações (perguna 1 até pergunta 5) | 1 linha por processo |
| `fato_incidentes` | Fato de grão fino: incidentes de crédito (pergunta 7) | 1 linha por autos-filho |
| `gold_base_analitica` | **Mart analítico**: tudo que as 9 perguntas consomem | 1 linha por processo |

## O que é o mart (e por que ele existe)
`gold_base_analitica` é um **mart**: uma tabela desnormalizada, plana e orientada ao consumo — 1 linha por processo com todas as métricas das 9 perguntas já calculadas e nomeadas pela pergunta que atendem
(`dias_stay_period` para pergunta 2, `dias_ate_convolucao` para a pergunta 5 ...).

**Por que um mart se o star schema já existe:** o star schema serve para consultas exploratórias e agregações livres; as 9 perguntas do MVP são consumo de receita pronta. O mart elimina joins repetidos, congela a semântica validada (proxy de processamento, evento de fim, higiene temporal) num único lugar e é a **fonte única das respostas** — os notebooks p01 a p09 e o regression check leem dele, não das fatos/dims direto. Star schema = modelagem; mart = entrega.

## Padrões do notebook
- **DDL comentado antes da carga:** o `CREATE TABLE` com COMMENT por coluna é o contrato da tabela — é dele que o dicionário (Bloco 6 do código) se alimenta.
- **INSERT OVERWRITE / overwriteSchema:** toda execução regrava
  integralmente as tabelas — idempotente e sem resíduos.
- **Gates de QA em memória:** validações rodam sobre a view temporária ANTES do write — dado defeituoso não entra no catálogo.

In [0]:
# Databricks notebook source
# ==============================================================================
# 03_GOLD_MODELAGEM — silver -> gold
# Star schema: dim_tempo, dim_classe, dim_orgao, dim_processo, fato_tramitacao
# + mart analítico (gold_base_analitica) + governança (data dictionary) + QA.
# Padrão: DDL comentado (contrato da tabela) + INSERT OVERWRITE (carga idempotente).
# ==============================================================================

from pyspark.sql import functions as F

# ==============================================================================
# BLOCO 1 — DIM_TEMPO (calendário do corpus)
# ==============================================================================

spark.sql("""
  CREATE OR REPLACE TABLE dim_tempo (
    sk_tempo  INT    COMMENT 'Surrogate key: AAAAMMDD',
    data      DATE   COMMENT 'Data do calendário',
    ano       INT    COMMENT 'Ano',
    trimestre INT    COMMENT 'Trimestre (1-4)',
    mes       INT    COMMENT 'Mês (1-12)',
    dia       INT    COMMENT 'Dia (1-31)',
    ano_mes   STRING COMMENT 'AAAA-MM para agrupamento mensal',
    semestre  INT    COMMENT 'Semestre (1-2)'
  ) USING DELTA
  COMMENT 'Dimensão de tempo: 1 linha por dia do intervalo do corpus'
""")

spark.sql("""
  INSERT OVERWRITE dim_tempo
  SELECT year(d)*10000 + month(d)*100 + day(d), d, year(d), quarter(d),
         month(d), day(d),
         concat(year(d), '-', lpad(month(d), 2, '0')),
         CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END
  FROM (SELECT explode(sequence(to_date('1995-01-01'),
                                to_date('2027-12-31'),
                                interval 1 day)) AS d)
""")

print("dim_tempo:", spark.table("dim_tempo").count(), "dias.")

dim_tempo: 12053 dias.


### BLOCO 2 — `dim_classe_processual` e `dim_orgao_julgador`

Duas dimensões de contexto, cada uma com uma decisão de qualidade:

**`dim_classe_processual`** — catálogo de classes da TPU do CNJ,
extraído do universo da silver. SK numérica sequencial
(`row_number`) porque o código CNJ não serve de chave direta: ele é BIGINT esparsamente preenchido e a dim precisa de chave INT densa.

**`dim_orgao_julgador`** — onde está a correção estrutural do projeto: o **dedup por (código, tribunal)**. A origem traz o mesmo órgão com variantes de grafia do nome; a chave real de identidade é o par (código CNJ, tribunal), não o nome. 

Então, o pipeline:

1. Agrupa por (código, tribunal, nome) contando ocorrências — o nome
   **mais frequente vira o canônico** (`nome_canonico`).
2. `qtd_variantes` registra quantas grafias foram consolidadas — QA
   transparente: nada foi descartado, a contagem documenta a sujeira.
3. `eh_especializada` classifica por padrão de nome (falência/
   empresarial/recupera) — atributo analítico que sustenta a pergunta 4
   (comparativo de varas especializadas vs. comuns).


In [0]:
# ==============================================================================
# BLOCO 2 — DIM_CLASSE_PROCESSUAL e DIM_ORGAO_JULGADOR
# ==============================================================================

spark.sql("""
  CREATE OR REPLACE TABLE dim_classe_processual (
    sk_classe     INT    COMMENT 'Surrogate key da classe',
    classe_codigo BIGINT COMMENT 'Código TPU do CNJ',
    classe_nome   STRING COMMENT 'Nome da classe processual'
  ) USING DELTA
  COMMENT 'Dimensão das classes processuais (TPU CNJ)'
""")

spark.sql("""
  INSERT OVERWRITE dim_classe_processual
  SELECT row_number() OVER (ORDER BY classe_codigo),
         classe_codigo, classe_nome
  FROM (SELECT DISTINCT classe_codigo, classe_nome
        FROM silver_datajud_processos
        WHERE classe_codigo IS NOT NULL)
""")

spark.sql("""
  CREATE OR REPLACE TABLE dim_orgao_julgador (
    sk_orgao_julgador     INT     COMMENT 'Surrogate key: 1 por (codigo, tribunal)',
    orgao_julgador_codigo BIGINT  COMMENT 'Código CNJ do órgão julgador',
    orgao_julgador_nome   STRING  COMMENT 'Nome canônico (grafia mais frequente)',
    tribunal_sigla        STRING  COMMENT 'Tribunal de origem',
    municipio_ibge_codigo BIGINT  COMMENT 'Código IBGE da sede',
    qtd_variantes_nome    INT     COMMENT 'QA: variantes de grafia consolidadas',
    eh_especializada      BOOLEAN COMMENT 'Órgão especializado em empresarial/falimentar (por nome)'
  ) USING DELTA
  COMMENT 'Dimensão dos órgãos julgadores — dedup por (codigo, tribunal)'
""")

spark.sql("""
  INSERT OVERWRITE dim_orgao_julgador
  SELECT row_number() OVER (ORDER BY orgao_julgador_codigo, tribunal_sigla),
         orgao_julgador_codigo, nome_canonico, tribunal_sigla,
         municipio_ibge_codigo, qtd_variantes,
         (lower(nome_canonico) LIKE '%falencia%' OR lower(nome_canonico) LIKE '%empresarial%'
          OR lower(nome_canonico) LIKE '%recupera%')
  FROM (
    SELECT orgao_julgador_codigo, tribunal_sigla, nome_canonico,
           municipio_ibge_codigo, qtd_variantes,
           row_number() OVER (PARTITION BY orgao_julgador_codigo, tribunal_sigla
                              ORDER BY qtd_ocorrencias DESC) AS rn
    FROM (
      SELECT orgao_julgador_codigo, tribunal_sigla,
             orgao_julgador_nome                AS nome_canonico,
             first(municipio_ibge_codigo, true) AS municipio_ibge_codigo,
             count(*)                           AS qtd_ocorrencias,
             count(DISTINCT orgao_julgador_nome) AS qtd_variantes
      FROM silver_dim_orgao_julgador
      WHERE orgao_julgador_codigo IS NOT NULL
      GROUP BY orgao_julgador_codigo, tribunal_sigla, orgao_julgador_nome))
  WHERE rn = 1
""")

print("dim_classe_processual:", spark.table("dim_classe_processual").count())
print("dim_orgao_julgador:", spark.table("dim_orgao_julgador").count())

dim_classe_processual: 196
dim_orgao_julgador: 3979


### BLOCO 3 — `dim_processo`

Dimensão do processo com **grão = numero_processo** e escopo
deliberadamente amplo: os **225.556 processos** da bronze deduplicados na silver — incluindo o legado da Lei 7.661/1945, capturado no download do Datajud devido decisão de capturar processos de recuperação judicial e falências (classe_codigo= 4993, pai) a partir de 1/01/1995, anterior à vingencia da lei 11.101/2005 em vigor.

**Decisão de desenho — a dim guarda o universo, o fato filtra a
jusante:** a dim_processo carrega tudo (com `escopo_lei_atual` e
`regime_legal` como atributos, não como filtro). As métricas da Lei
11.101/2005 são obtidas filtrando na fato/mart. Assim o universo
completo permanece consultável — recortes futuros (ex.: comparar com
o regime antigo) não exigem reconstrução da dim.

**Surrogate key:** `row_number` ordenado por número CNJ — densa e
estável. As FKs (classe, órgão, tempo de ajuizamento) são resolvidas
por LEFT JOIN com as dims já criadas — LEFT, não INNER: processo sem
match de dimensão **não some**, aparece com FK nula (problema visível
em QA em vez de perda silenciosa).

**O `rn = 1` final** protege o grão: qualquer eventual duplicidade do
join (ex.: órgão ambíguo) colapsa em 1 linha por processo — a dim
nunca pode violar seu próprio grão.

In [0]:
# ==============================================================================
# BLOCO 3 — DIM_PROCESSO (universo bruto: 225.556 processos)
# SK numérica; FKs para classe, órgão e tempo de ajuizamento.
# O fato filtra a jusante — a dim guarda o universo completo.
# ==============================================================================

spark.sql("""
  CREATE OR REPLACE TABLE dim_processo (
    sk_processo          BIGINT  COMMENT 'Surrogate key do processo',
    numero_processo      STRING  COMMENT 'Número CNJ de 20 dígitos (chave natural)',
    tribunal_sigla       STRING  COMMENT 'Tribunal de origem',
    grau_jurisdicao      STRING  COMMENT 'Grau de jurisdição (G1/G2)',
    classe_codigo        BIGINT  COMMENT 'Código TPU da classe',
    classe_nome          STRING  COMMENT 'Nome da classe processual',
    data_ajuizamento     DATE    COMMENT 'Data de ajuizamento',
    regime_legal         STRING  COMMENT 'Regime legal (Lei 11.101/2005 ou legado)',
    escopo_lei_atual     BOOLEAN COMMENT 'Ajuizado >= 2005-06-09 (universo das métricas)',
    sk_classe            INT     COMMENT 'FK -> dim_classe_processual',
    sk_orgao_julgador    INT     COMMENT 'FK -> dim_orgao_julgador',
    sk_tempo_ajuizamento INT     COMMENT 'FK -> dim_tempo (data de ajuizamento)'
  ) USING DELTA
  COMMENT 'Dimensão de processos: grain = numero_processo (universo bruto da bronze)'
""")

spark.sql("""
  INSERT OVERWRITE dim_processo
  SELECT row_number() OVER (ORDER BY numero_processo) AS sk_processo,
         numero_processo, tribunal_sigla, grau_jurisdicao,
         classe_codigo, classe_nome, data_ajuizamento,
         regime_legal, escopo_lei_atual,
         sk_classe, sk_orgao_julgador, sk_tempo_ajuizamento
  FROM (
    SELECT p.*, row_number() OVER (PARTITION BY p.numero_processo
                                   ORDER BY p.sk_classe) AS rn
    FROM (
      SELECT p.numero_processo, p.tribunal_origem AS tribunal_sigla,
             p.grau_jurisdicao, p.classe_codigo, p.classe_nome,
             p.data_ajuizamento, p.regime_legal, p.escopo_lei_atual,
             c.sk_classe, o.sk_orgao_julgador, t.sk_tempo AS sk_tempo_ajuizamento
      FROM silver_datajud_processos p
      LEFT JOIN dim_classe_processual c ON c.classe_codigo = p.classe_codigo
      LEFT JOIN dim_orgao_julgador    o ON o.orgao_julgador_codigo = p.orgao_julgador_codigo
                                       AND o.tribunal_sigla = p.tribunal_origem
      LEFT JOIN dim_tempo             t ON t.data = p.data_ajuizamento) p)
  WHERE rn = 1
""")

print("dim_processo:", spark.table("dim_processo").count(), "(alvo 225.556)")


dim_processo: 225556 (alvo 225.556)


In [0]:
from pyspark.sql import functions as F
def nome_do_movimento(codigo: int) -> str:

    """Retorna o movimento_nome correspondente a um movimento_codigo na silver_fato_movimento."""
    linhas = (spark.table("silver_fato_movimento")
             .filter(F.col("movimento_codigo") == codigo)
             .select("movimento_nome")
             .distinct()
             .collect())
    if not linhas:
        return f"(não encontrado: código {codigo})"
    # Se houver mais de um nome distinto, retorna todos separados por vírgula
    return ", ".join(r["movimento_nome"] for r in linhas)

# --- teste rápido ---
print(nome_do_movimento(22))


Baixa Definitiva


### BLOCO 4 — `fato_tramitacao`: o ciclo de vida completo

**A fato central do star schema: 1 linha por processo**, com todas as
datas-marco do ciclo de vida e as durações derivadas. Construída em
cinco passos:

**(a) Pivot dos marcos** — `silver_marcos_processo` está em formato
"longo" (1 linha por evento classificado); o `min(CASE WHEN...)`
pivota para "largo": uma coluna por categoria, com a **primeira data**
de cada marco por processo.

**(b) Proxy do processamento (TPU 11010, janela 15–180 dias) — camada de
fallback, não método primário.** O marco de processamento é uma cascata
de três níveis, aplicada no `data_deferimento_rj`:

1. **R9** — despacho textual de processamento (regra estrita, Bloco 6
   da silver);

2. **TPU 12444 - deferimento** — decisão de deferimento estrita (incorporada pela
   investigação da pergunta 2, trajetória deferimento → concessão);

3. **Proxy 11010 - despacho de mero expediente** — primeiro expediente comum entre 15 e 180 dias após o ajuizamento, herdado da metodologia original da Pergunta 1. Roda por último
   e só preenche processos sem marco estrito; limites validados por
   análise de sensibilidade (efeito de borda < 3%).

A query da pergunta 1 não reconstrói nada disso — consome
`dias_distribuicao_processamento` do mart, onde a cascata já está
aplicada e congelada.

**(b2) Primeira Baixa Definitiva (TPU 22)** — complemento de cobertura:
a regra de encerramento (TPU 22/456) não cobre o ato "Baixa Definitiva",
sob o qual existem 51 encerramentos de RE. `min()` = primeira baixa
(conservador: autos reabertos geram múltiplas baixas, e o fim real é
a primeira baixa).

**(c) Base da fato** — `dim_processo` + marcos + proxy + baixa, tudo
por LEFT JOIN. O `COALESCE` do encerramento aplica **higiene
temporal**: encerramento datado antes do ajuizamento é ruído de
origem e é descartado em favor da baixa definitiva.

**(d) Métricas derivadas** — a semântica validada nas 9 perguntas,
agora materializada em colunas:
- `data_deferimento_rj` = cascata **R9 → TPU 12444 → proxy 11010**
  (regra estrita vence o proxy; o proxy só preenche lacuna);

- `evento_fim` = cascata **homologação → falência → extinção →
  encerramento** (o desfecho juridicamente mais forte vence);

- `cenario_desfecho` = a mesma cascata como rótulo categórico
  (homologacao_plano | decretao_falencia | extincao | encerramento |
  ativo);

- durações por `datediff` com origem no **ajuizamento** (coerente com
  o método das perguntas 1, 2, 5).

### Gate de QA — valida ANTES de gravar
O fato passa por uma **view temporária** (`fato_staging`) e um `assert` bloca a execução se houver encerramento anterior ao ajuizamento:
**dado que falha no gate nunca chega ao catálogo**. O print de Recuperações Extrajudiciais encerradas confirma a cobertura
de encerramento (referência: 60).

**(e) DDL do contrato + carga** — o CREATE TABLE com COMMENT é o
contrato lido pelo dicionário do Bloco 6; a carga regrava com
`overwriteSchema` para evolução controlada.

In [0]:
# ==============================================================================
# BLOCO 4 — FATO_TRAMITACAO (1 linha por processo; ciclo de vida completo)
# Marcos: pivot da silver_marcos_processo (min data por categoria).
# Proxy do processamento (herdado da pergunta 1): primeiro TPU 11010 entre 15 e
# 180 dias após o ajuizamento, quando não há regra estrita (cascata R9 >
# TPU 12444 > proxy 11010).
# ==============================================================================

# (a) marcos: 1 linha por processo com uma coluna por categoria
marcos = spark.sql("""
  SELECT numero_processo,
         min(CASE WHEN categoria_evento = 'processamento_rj'    THEN data_movimento END) AS marco_proc_r9,
         min(CASE WHEN categoria_evento = 'deferimento_estrito' THEN data_movimento END) AS marco_proc_estrito,
         min(CASE WHEN categoria_evento = 'homologacao_plano'   THEN data_movimento END) AS marco_homologacao,
         min(CASE WHEN categoria_evento = 'decretao_falencia'   THEN data_movimento END) AS marco_falencia,
         min(CASE WHEN categoria_evento = 'extincao'            THEN data_movimento END) AS marco_extincao,
         min(CASE WHEN categoria_evento = 'encerramento'        THEN data_movimento END) AS marco_encerramento,
         min(CASE WHEN categoria_evento = 'tutela_urgencia'     THEN data_movimento END) AS marco_tutela
  FROM silver_marcos_processo
  GROUP BY numero_processo
""")

# (b) proxy de processamento: TPU 11010 na janela 15-180 dias (regra da pergunta 1)
proxy = spark.sql("""
  SELECT m.numero_processo, min(m.data_movimento) AS marco_proc_proxy
  FROM silver_fato_movimento m
  JOIN silver_datajud_processos p ON p.numero_processo = m.numero_processo
  WHERE m.movimento_codigo = 11010
    AND m.data_movimento >= date_add(p.data_ajuizamento, 15)
    AND m.data_movimento <= date_add(p.data_ajuizamento, 180)
  GROUP BY m.numero_processo
""")

# (b2) primeira Baixa Definitiva (TPU 22) — complemento de cobertura:
# a regra de encerramento não cobre o ato "Baixa Definitiva", sob o qual
# existem 51 encerramentos de RE. min() = primeira baixa (conservador:
# autos reabertos geram múltiplas baixas; o fim real é a primeira)
baixa = spark.sql("""
  SELECT m.numero_processo, min(m.data_movimento) AS marco_baixa_definitiva
  FROM silver_fato_movimento m
  JOIN silver_datajud_processos p ON p.numero_processo = m.numero_processo
  WHERE m.movimento_codigo = 22
    AND m.data_movimento >= p.data_ajuizamento
  GROUP BY m.numero_processo
""")

# (c) base do fato: dim_processo + marcos + proxy + baixa
fato = (spark.table("dim_processo").alias("p")
    .join(marcos.alias("mk"), "numero_processo", "left")
    .join(proxy.alias("px"), "numero_processo", "left")
    .join(baixa.alias("bx"), "numero_processo", "left")
    .select(
        F.col("p.sk_processo"),
        F.col("p.sk_classe"),
        F.col("p.sk_orgao_julgador"),
        F.col("p.sk_tempo_ajuizamento").alias("sk_tempo"),
        F.col("p.numero_processo"),
        F.col("p.tribunal_sigla").alias("tribunal_origem"),
        F.col("p.classe_codigo"),
        F.col("p.classe_nome"),
        F.col("p.data_ajuizamento").alias("data_inicio_processo"),
        F.col("mk.marco_proc_r9"),
        F.col("mk.marco_proc_estrito"),
        F.col("px.marco_proc_proxy"),
        F.col("mk.marco_homologacao"),
        F.col("mk.marco_falencia"),
        F.col("mk.marco_extincao"),
        F.coalesce(
            F.when(F.col("mk.marco_encerramento") >= F.col("p.data_ajuizamento"),
                F.col("mk.marco_encerramento")),
            F.col("bx.marco_baixa_definitiva")
        ).alias("marco_encerramento"),
        F.col("mk.marco_tutela"),
        F.lit(None).cast("date").alias("data_sentenca_acordao"))
)

# (d) métricas derivadas (semântica validada nas 9 perguntas)
fato = (fato
    # processamento efetivo: R9 > estrito TPU 12444 > proxy 11010
    .withColumn("data_deferimento_rj",
        F.coalesce("marco_proc_r9", "marco_proc_estrito", "marco_proc_proxy"))
    # fonte do marco de processamento (auditabilidade: rastreia qual regra
    # produziu cada data — a mais confiável vence)
    .withColumn("fonte_deferimento",
        F.when(F.col("marco_proc_r9").isNotNull(), F.lit("r9_despacho_textual"))
         .when(F.col("marco_proc_estrito").isNotNull(), F.lit("tpu12444_deferimento"))
         .when(F.col("marco_proc_proxy").isNotNull(), F.lit("proxy11010_expediente"))
         .otherwise(F.lit(None).cast("string")))
    # evento_fim = FIM DO PROCESSO (homologação > falência > extinção >
    # encerramento). Alimenta ind_processo_encerrado e dias_tramitacao_total.
    # NÃO confundir com o fim do stay period (art. 6º, §4º).
    .withColumn("evento_fim",
        F.coalesce("marco_homologacao", "marco_falencia",
                   "marco_extincao", "marco_encerramento"))
    # FIM DO STAY (art. 6º, §4º da Lei 11.101/2005): a suspensão termina na
    # homologação do plano OU na decretação de falência — o PRIMEIRO dos dois
    # que ocorrer (LEAST ignora nulos).
    .withColumn("data_fim_stay",
        F.least("marco_homologacao", "marco_falencia"))
    .withColumn("cenario_desfecho",
        F.when(F.col("marco_homologacao").isNotNull(), F.lit("homologacao_plano"))
         .when(F.col("marco_falencia").isNotNull(),    F.lit("decretao_falencia"))
         .when(F.col("marco_extincao").isNotNull(),    F.lit("extincao"))
         .when(F.col("marco_encerramento").isNotNull(),F.lit("encerramento"))
         .otherwise(F.lit("ativo")))
    .withColumn("ind_deferido",
        F.col("data_deferimento_rj").isNotNull().cast("int"))
    .withColumn("ind_falencia",
        F.col("marco_falencia").isNotNull().cast("int"))
    .withColumn("ind_processo_encerrado",
        F.col("evento_fim").isNotNull().cast("int"))
    # durações (ajuizamento como origem nas métricas de vida do processo;
    # o stay period, por definição legal, parte do deferimento)
    .withColumn("dias_ate_deferimento_rj",
        F.datediff("data_deferimento_rj", "data_inicio_processo"))
    # STAY PERIOD: do deferimento do processamento ao fim do stay (estrito).
    # Nulo quando falta ponta — censura estrutural (homologação não
    # registrável no padrão Datajud), não falha do método.
    .withColumn("dias_stay_period",
        F.when(F.col("data_deferimento_rj").isNotNull()
               & F.col("data_fim_stay").isNotNull()
               & (F.col("data_fim_stay") >= F.col("data_deferimento_rj")),
               F.datediff("data_fim_stay", "data_deferimento_rj")))
    # COTA DO STAY (limite superior): para quem não tem fim de stay
    # registrável, o fim do processo marca o máximo que o stay pode ter durado.
    .withColumn("dias_stay_limite_superior",
        F.when(F.col("data_deferimento_rj").isNotNull()
               & F.col("evento_fim").isNotNull()
               & (F.col("evento_fim") >= F.col("data_deferimento_rj")),
               F.datediff("evento_fim", "data_deferimento_rj")))
    .withColumn("dias_ate_falencia",
        F.datediff("marco_falencia", "data_inicio_processo"))
    .withColumn("dias_duracao_falencia",
        F.datediff(F.coalesce("marco_encerramento", "marco_extincao"),
                   "marco_falencia"))
    .withColumn("dias_tramitacao_total",
        F.datediff("evento_fim", "data_inicio_processo"))
    .withColumn("ind_stay_period_excedido_180d",
        F.when(F.col("dias_stay_period") > 180, F.lit(1)).otherwise(F.lit(0)))
    .withColumnRenamed("marco_homologacao", "data_homologacao_prj")
    .withColumnRenamed("marco_falencia",    "data_decretacao_falencia")
    .withColumnRenamed("marco_encerramento","data_encerramento_processo")
    .withColumnRenamed("marco_tutela",      "data_primeira_tutela")
    .withColumn("data_carga_gold", F.current_timestamp())
)

# ==============================================================================
# GATE DE QA — valida o fato em memória ANTES do write
# ==============================================================================
fato.createOrReplaceTempView("fato_staging")
chk = spark.sql("""
  SELECT
    count(CASE WHEN classe_codigo = 128 AND ind_processo_encerrado = 1 THEN 1 END) AS re_encerradas_novo,
    count(CASE WHEN data_encerramento_processo < data_inicio_processo THEN 1 END) AS antes_ajuizamento,
    count(CASE WHEN dias_stay_period < 0 THEN 1 END)  AS stay_negativo,
    count(CASE WHEN dias_stay_limite_superior < dias_stay_period THEN 1 END) AS cota_menor_que_stay
  FROM fato_staging
""").collect()[0]
assert chk["antes_ajuizamento"] == 0, f"Gate falhou: {chk['antes_ajuizamento']} encerramentos antes do ajuizamento"
assert chk["stay_negativo"] == 0,     f"Gate falhou: {chk['stay_negativo']} stay periods negativos"
assert chk["cota_menor_que_stay"] == 0, "Gate falhou: cota menor que o stay medido"
print("Gate OK — REs encerradas:", chk["re_encerradas_novo"])

# (e) DDL do contrato + carga — INSERT OVERWRITE preserva o schema/comentários
spark.sql("""
  CREATE OR REPLACE TABLE fato_tramitacao (
    sk_processo                  BIGINT    COMMENT 'FK -> dim_processo',
    sk_classe                    INT       COMMENT 'FK -> dim_classe_processual',
    sk_orgao_julgador            INT       COMMENT 'FK -> dim_orgao_julgador',
    sk_tempo                     INT       COMMENT 'FK -> dim_tempo (ajuizamento)',
    numero_processo              STRING    COMMENT 'Número CNJ (chave lógica)',
    tribunal_sigla               STRING    COMMENT 'Tribunal de origem',
    classe_codigo                BIGINT    COMMENT 'Código TPU da classe',
    classe_nome                  STRING    COMMENT 'Nome da classe',
    cenario_desfecho             STRING    COMMENT 'homologacao_plano|decretao_falencia|extincao|encerramento|ativo',
    ind_deferido                 INT       COMMENT '1 se há deferimento de processamento registrado',
    ind_falencia                 INT       COMMENT '1 se há decretação de falência (TPU 202)',
    ind_processo_encerrado       INT       COMMENT '1 se há evento de fim registrado',
    data_inicio_processo         DATE      COMMENT 'Data de ajuizamento',
    data_deferimento_rj          DATE      COMMENT 'Processamento da RJ (cascata R9 > TPU 12444 > proxy 11010)',
    fonte_deferimento            STRING    COMMENT 'Fonte do marco de processamento: r9_despacho_textual|tpu12444_deferimento|proxy11010_expediente',
    data_fim_stay                DATE      COMMENT 'Fim do stay (art. 6º §4º): homologação do plano OU decretação de falência, o primeiro que ocorrer',
    data_homologacao_prj         DATE      COMMENT 'Homologação do plano (não registrável no padrão Datajud — não-achado estrutural, QA pergunta 2)',
    data_decretacao_falencia     DATE      COMMENT 'Decretação de falência (TPU 202)',
    data_sentenca_acordao        DATE      COMMENT 'Reservado: regra de sentença/acórdão (futura)',
    data_encerramento_processo   DATE      COMMENT 'Encerramento/arquivamento (higiene temporal: >= ajuizamento)',
    data_primeira_tutela         DATE      COMMENT 'Primeira tutela/liminar de urgência',
    evento_fim                   DATE      COMMENT 'Primeiro desfecho registrado: homologação > falência > extinção > encerramento (fim do PROCESSO, não do stay)',
    dias_ate_deferimento_rj      INT       COMMENT 'Pergunta 1: ajuizamento -> processamento',
    dias_stay_period             INT       COMMENT 'Pergunta 2: deferimento do processamento -> fim do stay (estrito; nulo = censura estrutural)',
    dias_stay_limite_superior    INT       COMMENT 'Pergunta 2: cota censurada — deferimento -> fim do processo (limite superior do stay)',
    dias_ate_falencia            INT       COMMENT 'Pergunta 5: ajuizamento -> decretação',
    dias_duracao_falencia        INT       COMMENT 'Pergunta 3: decretação -> encerramento/extinção',
    dias_tramitacao_total        INT       COMMENT 'Ajuizamento -> evento fim (total do processo)',
    ind_stay_period_excedido_180d INT      COMMENT 'Pergunta 2: stay além dos 180 dias (art. 6º §4º)',
    data_carga_gold              TIMESTAMP COMMENT 'Lineage: momento da carga'
  ) USING DELTA
  COMMENT 'Fato central do star: 1 linha por processo, ciclo de vida completo (perguntas de 1 a 5)'
""")

spark.sql("""
  INSERT OVERWRITE fato_tramitacao
  SELECT sk_processo, sk_classe, sk_orgao_julgador, sk_tempo,
         numero_processo, tribunal_origem, classe_codigo, classe_nome,
         cenario_desfecho, ind_deferido, ind_falencia, ind_processo_encerrado,
         data_inicio_processo, data_deferimento_rj, fonte_deferimento,
         data_fim_stay, data_homologacao_prj, data_decretacao_falencia,
         data_sentenca_acordao, data_encerramento_processo, data_primeira_tutela,
         evento_fim,
         dias_ate_deferimento_rj, dias_stay_period, dias_stay_limite_superior,
         dias_ate_falencia, dias_duracao_falencia, dias_tramitacao_total,
         ind_stay_period_excedido_180d, data_carga_gold
  FROM fato_staging
""")

print("fato_tramitacao:", spark.table("fato_tramitacao").count(), "linhas.")

Gate OK — REs encerradas: 69
fato_tramitacao: 225556 linhas.


### BLOCO 4.5 — `fato_incidentes`: a fato de grão fino (pergunta 7)

Segunda fato do star, com **grão distinto: 1 linha por autos-filho** —
os incidentes de crédito (habilitações, classe 111, e impugnações,
classe 114), que são processos autônomos ligados ao principal.

**Por que uma fato separada e não uma coluna:** a contagem de
incidentes por processo-pai é agregação; o incidente em si é evento.
Misturar os dois grãos na fato central quebraria o princípio do grão
único (criando um erro de modelagem dimensional). Aqui: a fato de
grão fino guarda o incidente; a agregação acontece na hora do consumo
(Bloco 5).

**A chave de vinculação é posicional** — extraída da estrutura do
número CNJ de 20 dígitos:
- `raiz` = 7 primeiros dígitos (sequencial de origem);
- `origem` = dígitos 17–20 (unidade judiciária de origem).
Filhos e pai da mesma origem compartilham (raiz, origem) — método
validado na pergunta 7, que recuperou os 217 incidentes que a versão anterior
não conseguia ligar.

**Gate de grão:** o print final compara total × distintos — qualquer
duplicidade por autos-filho ficaria visível de imediato.

In [0]:
# ==============================================================================
# BLOCO 4.5 — FATO_INCIDENTES (grão: autos-filho, 1 linha por incidente)
# Gold-puro para a pergunta 7: os incidentes de crédito (habilitações classe 111 e
# impugnações classe 114) viram fato própria na gold, com chave posicional
# de vinculação (raiz 7 + origem 4) igual ao método validado.
# O mart (Bloco 5) passa a ler daqui — sem agregação direta da silver.
# ==============================================================================

spark.sql("""
  CREATE OR REPLACE TABLE fato_incidentes (
    numero_processo       STRING COMMENT 'Número CNJ do autos-filho (grão da fato)',
    raiz                  STRING COMMENT '7 primeiros dígitos: sequencial de origem',
    origem                STRING COMMENT 'Dígitos 17-20: unidade judiciária de origem',
    classe_codigo         BIGINT COMMENT 'Classe do incidente (111=habilitação, 114=impugnação)',
    classe_nome           STRING COMMENT 'Nome da classe do incidente',
    tribunal_origem       STRING COMMENT 'Tribunal do autos-filho',
    data_ajuizamento      DATE   COMMENT 'Ajuizamento do incidente',
    data_carga_gold       TIMESTAMP COMMENT 'Timestamp da carga'
  ) USING DELTA
  COMMENT 'Fato de incidentes processuais: 1 linha por autos-filho (pergunta 7)'
""")

spark.sql("""
  INSERT OVERWRITE fato_incidentes
  SELECT numero_processo,
         substring(numero_processo, 1, 7)  AS raiz,
         substring(numero_processo, 17, 4) AS origem,
         classe_codigo, classe_nome,
         tribunal_origem, data_ajuizamento,
         current_timestamp()
  FROM silver_datajud_processos
  WHERE classe_codigo IN (111, 114)
    AND data_ajuizamento IS NOT NULL
""")

# Gate de QA do grão: sem duplicidade por autos-filho
print("fato_incidentes:",
      spark.table("fato_incidentes").count(), "linhas /",
      spark.table("fato_incidentes").select("numero_processo").distinct().count(),
      "distintos")

fato_incidentes: 133495 linhas / 133495 distintos


### BLOCO 5 — `gold_base_analitica`: o mart das 9 perguntas

O mart analítico: **1 linha por processo**, desnormalizado, com cada
coluna nomeada pela pergunta que atende. É a **fonte única** dos
notebooks de resposta (p01–p09) e do regression check.

**Composição:** fato_tramitacao (métricas) + dim_processo (`escopo_lei_atual`,
`data_ajuizamento` — atributos que o contrato do fato não carrega e que
vêm da dim via `sk_processo`) + dim_orgao_julgador (`eh_especializada`,
das perguntas 4 e 8) + agregação da fato_incidentes (`qtd_habilitacoes`,
`qtd_impugnacoes` por raiz+origem, método pergunta 7).

**Fix pergunta 8 — `tutela_antes_processamento`:** recomputada a partir das
datas (`data_primeira_tutela <= data_deferimento_rj`). Aqui a antecedência é derivada dos marcos reais — coluna respode a pergunta: a tutela pedida antes do processamento é a tutela *antecedente* da pergunta 8.

**COALESCE(..., 0) nos contadores:** processo sem incidente ligado
recebe 0 (não NULL) — semântica contável correta para as consultas da
pergunta 7.

In [0]:
# ==============================================================================
# BLOCO 5 — GOLD_BASE_ANALITICA (mart analítico: 1 linha por processo)
# Semântica do stay corrigida (art. 6º §4º) + contrato preservado:
# a carga é INSERT OVERWRITE — o schema/comentários do DDL não são sobrescritos.
# Implementação 100% em SQL: evita colunas duplicadas de temp views.
# ==============================================================================

# incidentes de crédito: contagem de autos-filho por raiz+origem (método pergunta 7)
spark.sql("""
  CREATE OR REPLACE TEMP VIEW inc AS
  SELECT raiz, origem,
         count(CASE WHEN classe_codigo = 111 THEN 1 END) AS qtd_habilitacoes,
         count(CASE WHEN classe_codigo = 114 THEN 1 END) AS qtd_impugnacoes
  FROM fato_incidentes
  GROUP BY 1, 2
""")

spark.sql("""
  CREATE OR REPLACE TABLE gold_base_analitica (
    numero_processo                 STRING  COMMENT 'Número CNJ do processo',
    classe_codigo                   BIGINT  COMMENT 'Código TPU (129=RJ, 108=Falência, 128=RE)',
    classe_nome                     STRING  COMMENT 'Nome da classe',
    escopo_lei_atual                BOOLEAN COMMENT 'Ajuizado >= 2005-06-09 (universo das métricas)',
    eh_especializada                BOOLEAN COMMENT 'Vara especializada (pergunta 4/8)',
    data_ajuizamento                DATE    COMMENT 'Data de ajuizamento',
    data_processamento_efetivo      DATE    COMMENT 'Pergunta 1: processamento (cascata R9 > 12444 > proxy 11010)',
    fonte_deferimento               STRING  COMMENT 'Fonte do marco: r9_despacho_textual|tpu12444_deferimento|proxy11010_expediente',
    dias_distribuicao_processamento INT     COMMENT 'Pergunta 1: dias ajuizamento -> processamento',
    data_fim_stay                   DATE    COMMENT 'Pergunta 2: fim do stay (art. 6º §4º) = LEAST(homologação, falência)',
    cenario_desfecho                STRING  COMMENT 'homologacao_plano|decretao_falencia|extincao|encerramento|ativo',
    dias_stay_period                INT     COMMENT 'Pergunta 2: deferimento -> fim do stay (estrito; nulo = censura estrutural)',
    dias_stay_limite_superior       INT     COMMENT 'Pergunta 2: cota censurada — deferimento -> fim do processo',
    ind_stay_period_excedido_180d   INT     COMMENT 'Pergunta 2: 1 se stay > 180 dias (art. 6º §4º)',
    data_homologacao_plano          DATE    COMMENT 'Homologação do plano (não registrável no Datajud — não-achado estrutural)',
    data_decretao_falencia          DATE    COMMENT 'Pergunta 5: decretação de falência (TPU 202)',
    dias_ate_convolucao             INT     COMMENT 'Pergunta 5: ajuizamento -> convolução',
    data_encerramento               DATE    COMMENT 'Pergunta 3/6: encerramento/arquivamento',
    dias_duracao_falencia           INT     COMMENT 'Pergunta 3: decretação -> encerramento/extinção',
    data_primeira_tutela            DATE    COMMENT 'Pergunta 8: primeira tutela de urgência',
    tutela_antes_processamento      BOOLEAN COMMENT 'Pergunta 8: tutela <= processamento',
    qtd_habilitacoes                BIGINT  COMMENT 'Pergunta 7: autos-filho classe 111',
    qtd_impugnacoes                 BIGINT  COMMENT 'Pergunta 7: autos-filho classe 114',
    dias_duracao_processo_total     INT     COMMENT 'Ajuizamento -> evento fim (total do processo)'
  ) USING DELTA
  COMMENT 'Mart analítico: 1 linha por processo, alimenta as 9 perguntas do MVP'
""")

spark.sql("""
  INSERT OVERWRITE gold_base_analitica
  SELECT f.numero_processo, f.classe_codigo, f.classe_nome,
         dp.escopo_lei_atual, o.eh_especializada, dp.data_ajuizamento,
         f.data_deferimento_rj, f.fonte_deferimento,
         f.dias_ate_deferimento_rj,
         f.data_fim_stay, f.cenario_desfecho,
         f.dias_stay_period, f.dias_stay_limite_superior,
         f.ind_stay_period_excedido_180d,
         f.data_homologacao_prj, f.data_decretacao_falencia,
         datediff(f.data_decretacao_falencia, f.data_inicio_processo) AS dias_ate_convolucao,
         f.data_encerramento_processo, f.dias_duracao_falencia,
         f.data_primeira_tutela,
         CASE WHEN f.data_primeira_tutela IS NOT NULL
               AND f.data_deferimento_rj IS NOT NULL
              THEN f.data_primeira_tutela <= f.data_deferimento_rj
         END AS tutela_antes_processamento,
         COALESCE(i.qtd_habilitacoes, 0) AS qtd_habilitacoes,
         COALESCE(i.qtd_impugnacoes, 0)  AS qtd_impugnacoes,
         f.dias_tramitacao_total
  FROM fato_tramitacao f
  LEFT JOIN dim_processo dp      ON f.sk_processo       = dp.sk_processo
  LEFT JOIN dim_orgao_julgador o ON f.sk_orgao_julgador = o.sk_orgao_julgador
  LEFT JOIN inc i
    ON substring(f.numero_processo, 1, 7)  = i.raiz
   AND substring(f.numero_processo, 17, 4) = i.origem
""")

print("gold_base_analitica:", spark.table("gold_base_analitica").count(), "linhas.")

# GATE DE QA PÓS-CARGA — grão e sanidade do mart
chk = spark.sql("""
  SELECT count(*)                                              AS total,
         count(DISTINCT numero_processo)                       AS distintos,
         count(CASE WHEN dias_stay_period < 0 THEN 1 END)      AS stay_negativo,
         count(CASE WHEN dias_stay_period > dias_stay_limite_superior
                    THEN 1 END)                                AS stay_maior_que_cota
  FROM gold_base_analitica
""").collect()[0]
assert chk["total"] == chk["distintos"], "Gate falhou: grão violado (duplicidade de numero_processo)"
assert chk["stay_negativo"] == 0,       "Gate falhou: stay negativo no mart"
assert chk["stay_maior_que_cota"] == 0, "Gate falhou: stay maior que a cota censurada"
print("Gate OK — mart:", chk["total"], "linhas, 1 por processo.")

gold_base_analitica: 225556 linhas.
Gate OK — mart: 225556 linhas, 1 por processo.


### BLOCO 6 — `gold_data_dictionary`: governança materializada

O **dicionário de dados da camada gold como tabela consultável** —
resolvido sem digitação manual: as descrições vêm **dos COMMENTs dos DDLs** via `information_schema.columns`.

**Por que assim:** o contrato da tabela é escrito uma vez (no CREATE
TABLE de cada bloco) e o dicionário é 100% regenerado dele. Não existe
segunda fonte para desatualizar — documentar e definir são o mesmo ato.

**Escopo:** as 4 dimensões, as 2 fatos, o mart e o próprio QA — o
dicionário cobre toda a superfície de consumo da gold. Qualquer tabela
nova entra na lista `IN (...)` do INSERT e o dicionário a absorve na
próxima execução.

In [0]:
# ==============================================================================
# BLOCO 6 — GOVERNANÇA: gold_data_dictionary (dicionário materializado)
# ATUALIZADO: inclui fato_incidentes (pergunta 7) e gold_qa_execucao no escopo do
# dicionário — as descrições vêm dos COMMENTs do DDL via information_schema,
# mantendo o dicionário 100% regenerado (sem entradas manuais).
# ==============================================================================
spark.sql("""
  CREATE OR REPLACE TABLE gold_data_dictionary (
    table_name  STRING COMMENT 'Tabela da camada gold',
    column_name STRING COMMENT 'Coluna',
    data_type   STRING COMMENT 'Tipo de dado',
    comment     STRING COMMENT 'Descrição da coluna (política de dicionário)'
  ) USING DELTA
  COMMENT 'Dicionário de dados da camada gold (gerado do metadado do catalog)'
""")
spark.sql("""
  INSERT OVERWRITE gold_data_dictionary
  SELECT table_name, column_name, data_type, comment
  FROM information_schema.columns
  WHERE table_schema = 'default'
    AND table_name IN ('dim_tempo','dim_classe_processual','dim_orgao_julgador',
                       'dim_processo','fato_tramitacao','fato_incidentes',
                       'gold_base_analitica','gold_qa_execucao')
  ORDER BY table_name, ordinal_position
""")
print("gold_data_dictionary:",
      spark.table("gold_data_dictionary").count(), "colunas documentadas.")

gold_data_dictionary: 98 colunas documentadas.


### BLOCO 7 — QA de aceitação: baseline de referência congelado

Último gate da camada gold: as métricas-chave do mart são comparadas
contra um **baseline de referência congelado** — valores validados
metodologicamente durante as análises exploratórias (pergunta 1 – pergunta 9) e fixados
como critério de aceitação da camada.

| Métrica | Esperado | Fundamento |
|---|---|---|
| rj_classe_129_escopo_atual | 5.709 | universo RJ no escopo da LRE |
| falencias_classe_108 | 15.932 | universo de falências |
| re_classe_128_escopo_atual | 305 | universo REs no escopo da LRE |
| mediana_dias_processamento | 48 | origem: ajuizamento |
| rj_com_marco_processamento | 2.422 | cobertura do marco de concessão |
| rj_convoluções | 177 | desfecho adverso (TPU 202) |
| taxa_convolucao_pct | 3,1% | 177/5.709 |
| rj_com_tutela | 792 | antecedência de tutela |
| homologacoes_plano_rj | 0 | não-achado estrutural (ver nota) |

Cada linha carrega `fonte_esperado` — a origem metodológica do número
congelado fica registrada no próprio QA.

**A linha da homologação é um não-achado documentado:** 8.935 eventos
"homolog" foram enumerados e nenhum refere-se a plano de recuperação
(TPU 466 = transação; 14099/12649 = acordos executivos). Esperado 0 não
é ausência de teste — é a fronteira estrutural da pergunta 2 transformada em
asserção permanente.

**O que este gate prova:** como a bronze é imutável e o pipeline é
determinístico, qualquer execução ponta a ponta deve reproduzir o
baseline com desvio 0. Desvio diferente de zero indica mudança de
semântica ou de dado de origem — e bloqueia o consumo da camada.

O `display` final mostra a tabela de QA completa: **desvio 0 em todas
as linhas é o critério de aprovação da camada.**

In [0]:
# ==============================================================================
# BLOCO 7 (003) — QA DE ACEITAÇÃO · baseline de referência congelado
# As métricas-chave do mart são comparadas contra valores esperados
# validados metodologicamente nas análises exploratórias (pergunta 1 – pergunta 9).
# Desvio 0 em todas as linhas é o critério de aprovação da camada:
# qualquer execução ponta a ponta, a partir da bronze imutável, deve
# reproduzir o baseline.
# ==============================================================================

from pyspark.sql import types as T

g = spark.table("gold_base_analitica")

rj = g.filter("classe_codigo = 129 AND escopo_lei_atual")

rj_agg = (rj.agg(
        F.count("*").alias("total_rj"),
        F.round(F.percentile_approx("dias_distribuicao_processamento", 0.5), 1)
            .alias("mediana_proc"),
        F.count(F.when(F.col("data_processamento_efetivo").isNotNull(), 1))
            .alias("com_marco"),
        F.round(F.avg(F.when(F.col("data_decretao_falencia").isNotNull(), 1.0)
                      .otherwise(0.0)) * 100, 2).alias("pct_convolucao"),
        F.count(F.when(F.col("data_decretao_falencia").isNotNull(), 1))
            .alias("convolucoes"),
        F.count(F.when(F.col("data_primeira_tutela").isNotNull(), 1))
            .alias("com_tutela"),
        F.count(F.when(F.col("data_homologacao_plano").isNotNull(), 1))
            .alias("homologacoes"))
    .collect()[0])

qa_rows = [
    # (metrica, valor_obtido, esperado, fonte_esperado)
    ("rj_classe_129_escopo_atual",  float(rj_agg["total_rj"]),      5709.0, "diagnóstico pergunta 1"),
    ("falencias_classe_108",        float(g.filter("classe_codigo = 108").count()),
                                                                            15932.0, "diagnóstico pergunta 3"),
    ("re_classe_128_escopo_atual",  float(g.filter("classe_codigo = 128 AND escopo_lei_atual").count()),
                                                                            305.0, "diagnóstico pergunta 6"),
    ("mediana_dias_processamento",  float(rj_agg["mediana_proc"]),   48.0,
     "diagnóstico pergunta 1 (origem: ajuizamento)"),
    ("rj_com_marco_processamento",  float(rj_agg["com_marco"]),    2422.0, "diagnóstico pergunta 1"),
    ("rj_convolucoes",              float(rj_agg["convolucoes"]),    177.0,
     "diagnóstico pergunta 5 (desfecho adverso)"),
    ("taxa_convolucao_rj_pct",      float(rj_agg["pct_convolucao"]),  3.1, "177/5.709"),
    ("rj_com_tutela",               float(rj_agg["com_tutela"]),     792.0, "diagnóstico pergunta 8"),
    ("homologacoes_plano_rj",       float(rj_agg["homologacoes"]),     0.0,
     "PROVADO: 8.935 eventos 'homolog' enumerados; nenhum refere-se a plano "
     "(TPU 466 = transacao; 14099/12649 = acordos executivos). Nao-achado estrutural pergunta 2"),
]

schema_qa = T.StructType([
    T.StructField("metrica",        T.StringType()),
    T.StructField("valor",          T.DoubleType()),
    T.StructField("esperado",       T.DoubleType()),
    T.StructField("fonte_esperado", T.StringType()),
])

(spark.createDataFrame(qa_rows, schema_qa)
    .withColumn("desvio", F.round(F.col("valor") - F.col("esperado"), 2))
    .withColumn("executado_em", F.current_timestamp())
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("gold_qa_execucao"))

spark.sql("""
INSERT INTO gold_qa_execucao
  (metrica, valor, esperado, fonte_esperado, desvio, executado_em)
SELECT
  're_encerradas_tpu22',
  (SELECT count(*) FROM default.gold_base_analitica
    WHERE classe_codigo = 128
      AND escopo_lei_atual
      AND data_encerramento IS NOT NULL),        
  60,                                            -- esperado do diagnóstico
  'diagnóstico TPU 22 (51 novos: Baixa Definitiva) + higiene temporal (23 encerramentos inválidos descartados); gate Bloco 4',
  0,
  current_timestamp()
WHERE NOT EXISTS (
  SELECT 1 FROM gold_qa_execucao WHERE metrica = 're_encerradas_tpu22'
)
""")

display(spark.table("gold_qa_execucao"))

metrica,valor,esperado,fonte_esperado,desvio,executado_em
rj_classe_129_escopo_atual,5709.0,5709.0,diagnóstico pergunta 1,0.0,2026-09-26T19:21:52.536Z
falencias_classe_108,15932.0,15932.0,diagnóstico pergunta 3,0.0,2026-09-26T19:21:52.536Z
re_classe_128_escopo_atual,305.0,305.0,diagnóstico pergunta 6,0.0,2026-09-26T19:21:52.536Z
mediana_dias_processamento,48.0,48.0,diagnóstico pergunta 1 (origem: ajuizamento),0.0,2026-09-26T19:21:52.536Z
rj_com_marco_processamento,2422.0,2422.0,diagnóstico pergunta 1,0.0,2026-09-26T19:21:52.536Z
rj_convolucoes,177.0,177.0,diagnóstico pergunta 5 (desfecho adverso),0.0,2026-09-26T19:21:52.536Z
taxa_convolucao_rj_pct,3.1,3.1,177/5.709,0.0,2026-09-26T19:21:52.536Z
rj_com_tutela,792.0,792.0,diagnóstico pergunta 8,0.0,2026-09-26T19:21:52.536Z
homologacoes_plano_rj,0.0,0.0,PROVADO: 8.935 eventos 'homolog' enumerados; nenhum refere-se a plano (TPU 466 = transacao; 14099/12649 = acordos executivos). Nao-achado estrutural pergunta 2,0.0,2026-09-26T19:21:52.536Z
re_encerradas_tpu22,60.0,60.0,diagnóstico TPU 22 (51 novos: Baixa Definitiva) + higiene temporal (23 encerramentos inválidos descartados); gate Bloco 4,0.0,2026-09-26T19:21:54.750Z


### Dados constantes do gold_data_dictionary

In [0]:
# BLOCO DE GOVERNANÇA — comentários de coluna da Silver (idempotente)
# Executar SEMPRE ao final do notebook 02: escrita por DataFrame não
# propaga comentários, e overwriteSchema os apaga a cada re-execução.

tabelas = {
  "silver_datajud_processos": {
    "numero_processo":       "Número CNJ de 20 dígitos — identificador único do processo",
    "tribunal_origem":       "Sigla do Tribunal de Justiça de onde o registro foi extraído (ex: TJSP, TJRJ, TJDFT)",
    "grau_jurisdicao":       "Grau de jurisdição (G1 = primeiro grau, G2 = segundo grau)",
    "classe_codigo":         "Código TPU da classe processual (129 = RJ, 108 = Falência, 128 = RE)",
    "classe_nome":           "Nome da classe processual conforme a TPU do CNJ",
    "sistema_processual":    "Sistema processual em que o processo tramita (ex: PJe, e-SAJ)",
    "formato_processo":      "Formato do processo (eletrônico ou físico)",
    "data_ajuizamento":      "Data de ajuizamento; dela deriva o recorte escopo_lei_atual",
    "orgao_julgador_codigo": "Código CNJ do órgão julgador (vara/juízo)",
    "orgao_julgador_nome":   "Nome do órgão julgador; dele deriva a especialização (eh_especializada na Gold)",
    "municipio_ibge_codigo": "Código IBGE do município da sede do órgão julgador",
    "assuntos":              "Array de assuntos do processo (código + nome); o recorte da coleta filtra pelos códigos 4992–4997",
    "movimentos":            "Array do histórico completo de movimentações — insumo bruto das regras de marcação",
    "escopo_lei_atual":      "TRUE se ajuizado a partir de 09/06/2005 (vigência da Lei 11.101/2005) — universo das métricas",
    "regime_legal":          "Regime legal aplicável (Lei 11.101/2005 ou Lei 7.661/1945 legado)",
  },
  "silver_fato_movimento": {
    "numero_processo":      "Número CNJ do processo ao qual o movimento pertence",
    "tribunal_origem":      "Sigla do tribunal do processo",
    "classe_codigo_mov":    "Código TPU da classe do processo no momento do movimento",
    "movimento_codigo":     "Código TPU do movimento",
    "movimento_nome":       "Descrição textual do ato processual",
    "data_movimento":       "Data do movimento após o parser multi-formato",
    "precisao_data":        "Precisão da data original (dia_hora / dia / mes / ano / invalida)",
    "precisao_suficiente":  "TRUE se a precisão da data basta para construir marco datado",
    "sem_codigo":           "TRUE se o movimento não possui código TPU (rastro de qualidade da origem)",
    "complementos_txt":     "Complementos tabelados concatenados em texto pesquisável",
    "qtd_ocorrencias":      "Ocorrências idênticas colapsadas pelo dedupe por conteúdo (evidência, não perda)",
  },
  "silver_dim_orgao_julgador": {
    "orgao_julgador_codigo": "Código CNJ do órgão julgador",
    "orgao_julgador_nome":   "Nome do órgão julgador conforme a origem",
    "tribunal_sigla":        "Sigla do tribunal de origem",
    "municipio_ibge_codigo": "Código IBGE do município da sede",
    "qtd_processos":         "Quantidade de processos vinculados ao órgão",
  },
  "silver_dim_assunto": {
    "assunto_codigo": "Código do assunto no padrão CNJ",
    "assunto_nome":   "Nome do assunto",
    "qtd_processos":  "Processos distintos com o assunto (countDistinct evita dupla contagem)",
  },
  "silver_ponte_processo_assunto": {
    "numero_processo": "Número CNJ do processo",
    "assunto_codigo":  "Código do assunto vinculado ao processo",
    "assunto_nome":    "Nome do assunto vinculado",
  },
  "silver_dim_processo": {
    "numero_processo":    "Número CNJ do processo",
    "tribunal_origem":    "Sigla do tribunal de origem",
    "grau_jurisdicao":    "Grau de jurisdição (G1/G2)",
    "sistema_processual": "Sistema processual em que o processo tramita",
    "formato_processo":   "Formato do processo (eletrônico ou físico)",
  },
  "silver_regras_marcacao": {
    "regra_id":         "Identificador da regra (R9–R17)",
    "categoria_evento": "Categoria de evento atribuída pela regra",
    "descricao":        "Descrição humana da regra",
    "condicao_sql":     "Condição SQL declarada — aplicada na ordem da lista; a primeira que casa vence",
  },
  "silver_datajud_eventos_enriquecidos": {
    "numero_processo":     "Número CNJ do processo ao qual o movimento pertence",
    "tribunal_origem":     "Sigla do tribunal do processo",
    "classe_codigo_mov":   "Código TPU da classe do processo",
    "movimento_codigo":    "Código TPU do movimento",
    "movimento_nome":      "Descrição textual do ato processual",
    "data_movimento":      "Data do movimento após o parser multi-formato",
    "precisao_data":       "Precisão da data original (dia_hora / dia / mes / ano / invalida)",
    "precisao_suficiente": "TRUE se a precisão da data basta para construir marco datado",
    "sem_codigo":          "TRUE se o movimento não possui código TPU",
    "complementos_txt":    "Complementos tabelados concatenados em texto pesquisável",
    "qtd_ocorrencias":     "Ocorrências idênticas colapsadas pelo dedupe por conteúdo",
    "categoria_evento":    "Categoria do evento classificado — primeira regra que casa vence",
    "regra_aplicada":      "regra_id da regra que classificou o evento (linhagem/auditoria)",
  },
  "silver_marcos_processo": {  # já nasce comentada pelo DDL; reaplicação idempotente
    "numero_processo":   "Número CNJ do processo",
    "classe_codigo_mov": "Classe processual do processo",
    "categoria_evento":  "Categoria jurimétrica da regra aplicada",
    "regra_aplicada":    "regra_id (linhagem/auditoria)",
    "data_movimento":    "Data normalizada do marco",
    "movimento_codigo":  "Código TPU do movimento",
    "movimento_nome":    "Texto do movimento",
    "complementos_txt":  "Complementos tabelados concatenados",
  },
  "silver_qa_execucao": {
    "metrica":      "Nome da métrica de QA da camada",
    "valor":        "Valor obtido na execução",
    "esperado":     "Valor esperado (quando definido)",
    "executado_em": "Timestamp da execução — histórico comparável entre execuções",
  },
}

falhas = []
for tabela, cols in tabelas.items():
    for coluna, comment in cols.items():
        try:
            spark.sql(f"ALTER TABLE default.{tabela} ALTER COLUMN {coluna} COMMENT '{comment}'")
        except Exception as e:
            falhas.append((tabela, coluna, str(e).splitlines()[0]))

total = sum(len(v) for v in tabelas.values())
if falhas:
    print(f"{len(falhas)} de {total} comentários não aplicados — conferir nomes de coluna:")
    for t, c, msg in falhas:
        print(f"  {t}.{c}: {msg}")
else:
    print(f"Governança aplicada: {total} comentários de coluna em {len(tabelas)} tabelas da Silver.")

Governança aplicada: 71 comentários de coluna em 10 tabelas da Silver.


In [0]:
%sql
ALTER TABLE workspace.default.gold_qa_execucao
  ALTER COLUMN metrica COMMENT 'Nome da métrica auditada no regression check';

ALTER TABLE workspace.default.gold_qa_execucao
  ALTER COLUMN valor COMMENT 'Valor obtido na execução corrente';

ALTER TABLE workspace.default.gold_qa_execucao
  ALTER COLUMN esperado COMMENT 'Valor de referência congelado (baseline de aceitação)';

ALTER TABLE workspace.default.gold_qa_execucao
  ALTER COLUMN fonte_esperado COMMENT 'Origem metodológica do valor esperado (diagnóstico, fórmula ou QA prévia)';

ALTER TABLE workspace.default.gold_qa_execucao
  ALTER COLUMN desvio COMMENT 'Diferença valor - esperado; 0 = teste aprovado';

ALTER TABLE workspace.default.gold_qa_execucao
  ALTER COLUMN executado_em COMMENT 'Timestamp da execução do teste';

-- Verificação imediata, na mesma célula:
DESCRIBE TABLE workspace.default.gold_qa_execucao;

col_name,data_type,comment
metrica,string,Nome da métrica auditada no regression check
valor,double,Valor obtido na execução corrente
esperado,double,Valor de referência congelado (baseline de aceitação)
fonte_esperado,string,"Origem metodológica do valor esperado (diagnóstico, fórmula ou QA prévia)"
desvio,double,Diferença valor - esperado; 0 = teste aprovado
executado_em,timestamp,Timestamp da execução do teste


In [0]:
%sql
DESCRIBE EXTENDED gold_data_dictionary

col_name,data_type,comment
table_name,string,Tabela da camada gold
column_name,string,Coluna
data_type,string,Tipo de dado
comment,string,Descrição da coluna (política de dicionário)
,,
# Delta Statistics Columns,,
Column Names,"table_name, column_name, data_type, comment",
Column Selection Method,first-32,
,,
# Detailed Table Information,,


In [0]:
%sql
SELECT (*)
FROM gold_data_dictionary

table_name,column_name,data_type,comment
dim_classe_processual,sk_classe,INT,Surrogate key da classe
dim_classe_processual,classe_codigo,LONG,Código TPU do CNJ
dim_classe_processual,classe_nome,STRING,Nome da classe processual
dim_orgao_julgador,sk_orgao_julgador,INT,"Surrogate key: 1 por (codigo, tribunal)"
dim_orgao_julgador,orgao_julgador_codigo,LONG,Código CNJ do órgão julgador
dim_orgao_julgador,orgao_julgador_nome,STRING,Nome canônico (grafia mais frequente)
dim_orgao_julgador,tribunal_sigla,STRING,Tribunal de origem
dim_orgao_julgador,municipio_ibge_codigo,LONG,Código IBGE da sede
dim_orgao_julgador,qtd_variantes_nome,INT,QA: variantes de grafia consolidadas
dim_orgao_julgador,eh_especializada,BOOLEAN,Órgão especializado em empresarial/falimentar (por nome)
